In [127]:

import geopandas as gpd 
import pandas as pd
import fiona 
from dotenv import load_dotenv

load_dotenv()

%cat .env

RAW_DIR = 'data/raw/'
REF_DIR = 'data/processed/ref/'
GEOM_DIR = 'data/processed/geom/'
# variables du fichier .env
print(RAW_DIR)
print(REF_DIR)
print(GEOM_DIR)

# wget https://data.geopf.fr/telechargement/download/ADMIN-EXPRESS-COG/ADMIN-EXPRESS-COG_4-0__GPKG_LAMB93_FXX_2025-01-01/ADMIN-EXPRESS-COG_4-0__GPKG_LAMB93_FXX_2025-01-01.7z
# Documentation : https://geoservices.ign.fr/sites/default/files/2025-06/DL_ADMIN_EXPRESS_4-0.pdf
# File path: ../data/raw/territoires/ADE-COG_4-0_GPKG_LAMB93_FXX-ED2025-01-01.gpkg
file_name = "ADE-COG_4-0_GPKG_LAMB93_FXX-ED2025-01-01.gpkg"

file_path = '../' + RAW_DIR + 'territoires/' + file_name
print(f"File path: {file_path}")

%ls ../data/raw/territoires/"ADE-COG_4-0_GPKG_LAMB93_FXX-ED2025-01-01.gpkg"

DATA_DIR=data
RAW_DIR=${DATA_DIR}/raw
EXTRACTED_DIR=${DATA_DIR}/extracted
PROCESSED_DIR=${DATA_DIR}/processed

GEOM_DIR=${PROCESSED_DIR}/geom
REF_DIR=${PROCESSED_DIR}/ref
DATA_INDIC_DIR=${PROCESSED_DIR}/data
data/raw/
data/processed/ref/
data/processed/geom/
File path: ../data/raw/territoires/ADE-COG_4-0_GPKG_LAMB93_FXX-ED2025-01-01.gpkg
../data/raw/territoires/ADE-COG_4-0_GPKG_LAMB93_FXX-ED2025-01-01.gpkg*


In [128]:
ls -lh ../data/processed/geom/

total 203080
-rw-r--r--  1 jean-jacques  staff    24M Feb 28 18:55 app_communes_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff   1.4M Feb 28 18:57 collectivites_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff   934K Feb 25 19:10 collectivites_topo.json
-rw-r--r--  1 jean-jacques  staff    24M Feb 28 18:57 communes_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff    28M Feb 25 18:51 communes_topo.json
-rw-r--r--  1 jean-jacques  staff   1.6M Feb 28 18:57 departements_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff   935K Feb 25 18:59 departements_topo.json
-rw-r--r--  1 jean-jacques  staff   5.2M Feb 28 18:57 epci_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff   3.1M Feb 25 19:09 epci_topo.json
-rw-r--r--  1 jean-jacques  staff   617K Feb 28 18:57 regions_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff   457K Feb 25 18:59 regions_topo.json
-rw-r--r--  1 jean-jacques  staff   2.5M Feb 22 12:46 zone_emploi_geom.csv
-rw-r--r--  1

In [124]:
def add_coord(df):
    df = df.to_crs(4326)
    pts = df.geometry.representative_point()
    df["lon"] = pts.x
    df["lat"] = pts.y
    return df


In [129]:
layers = fiona.listlayers(file_path)

print("Layers trouvés :")
for layer in layers:
    print("-", layer)


Layers trouvés :
- canton
- arrondissement
- arrondissement_municipal
- chef_lieu_d_arrondissement
- chef_lieu_d_arrondissement_municipal
- chef_lieu_de_canton
- chef_lieu_de_collectivite_territoriale
- chef_lieu_de_commune
- chef_lieu_de_commune_associee_ou_deleguee
- chef_lieu_de_departement
- chef_lieu_d_epci
- chef_lieu_de_region
- collectivite_territoriale
- commune
- commune_associee_ou_deleguee
- departement
- epci
- region
- info_metadonnees
- layer_styles


In [130]:
communes = gpd.read_file(file_path, layer="commune") 
epci = gpd.read_file(file_path, layer="epci") 
departements = gpd.read_file(file_path, layer="departement") 
regions = gpd.read_file(file_path, layer="region") 
collectivites = gpd.read_file(file_path, layer="collectivite_territoriale")
arrond = gpd.read_file(file_path, layer="arrondissement_municipal")

communes.crs


<Projected CRS: EPSG:2154>
Name: RGF93 v1 / Lambert-93
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: France - onshore and offshore, mainland and Corsica (France métropolitaine including Corsica).
- bounds: (-9.86, 41.15, 10.38, 51.56)
Coordinate Operation:
- name: Lambert-93
- method: Lambert Conic Conformal (2SP)
Datum: Reseau Geodesique Francais 1993 v1
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [ ]:
# Simplification testée pour les cartes pydeck ds streamlit

def simplify_geom(gdf, tolerance=20):
    # Reprojection en Lambert 93 car pandas peut sans qu'on le sache le charger en WGS84 
    # la simplification est plus efficace dans un système métrique
    gdf = gdf.to_crs(2154)

    # Correction + simplification + correction
    gdf["geometry"] = (
        gdf["geometry"]
        .buffer(0)
        .simplify(tolerance, preserve_topology=False)
        .buffer(0)
    )

    # Reprojection finale en WGS84 pour le web
    return gdf.to_crs(4326)

In [131]:
# Pré calculer les points centraux des territoires
communes = add_coord(communes)
epci = add_coord(epci)
departements = add_coord(departements)
regions = add_coord(regions)
collectivites = add_coord(collectivites)
arrond = add_coord(arrond)

In [132]:
#ref_comm.to_crs(2154)
#ref_comm.to_crs(4326, inplace=True)
# Ajout latitude, longitude

communes.head()

,cleabs,nom_officiel,nom_officiel_en_majuscules,statut,code_insee,population,date_du_recensement,organisme_recenseur,code_insee_du_canton,code_insee_de_l_arrondissement,code_insee_du_departement,code_insee_de_la_region,code_siren,codes_siren_des_epci,code_postal,superficie_cadastrale,geometry,lon,lat
0,COMMUNE_0000000000001001,L'Abergement-Clémenciat,L'ABERGEMENT-CLEMENCIAT,Commune simple,01001,859,2022-01-01,INSEE,0108,012,01,84,210100012,200069193,01400,1590,"MULTIPOLYGON (((4.95841 46.15327, 4.95812 46.1...",4.934111,46.151674
1,COMMUNE_0000000000001002,L'Abergement-de-Varey,L'ABERGEMENT-DE-VAREY,Commune simple,01002,273,2022-01-01,INSEE,0101,011,01,84,210100020,240100883,01640,920,"MULTIPOLYGON (((5.4302 45.98277, 5.43012 45.98...",5.424073,46.007004
2,COMMUNE_0000000000001004,Ambérieu-en-Bugey,AMBERIEU-EN-BUGEY,Commune simple,01004,15554,2022-01-01,INSEE,0101,011,01,84,210100046,240100883,01500,2460,"MULTIPOLYGON (((5.40882 45.94206, 5.4085 45.94...",5.367885,45.957404
3,COMMUNE_0000000000001005,Ambérieux-en-Dombes,AMBERIEUX-EN-DOMBES,Commune simple,01005,1917,2022-01-01,INSEE,0122,012,01,84,210100053,200042497,01330,1590,"MULTIPOLYGON (((4.94298 45.97962, 4.94257 45.9...",4.914417,45.999403
4,COMMUNE_0000000000001006,Ambléon,AMBLEON,Commune simple,01006,114,2022-01-01,INSEE,0104,011,01,84,210100061,200040350,01300,590,"MULTIPOLYGON (((5.57083 45.75338, 5.57219 45.7...",5.593003,45.748372


In [133]:
communes.columns

Index(['cleabs', 'nom_officiel', 'nom_officiel_en_majuscules', 'statut',
       'code_insee', 'population', 'date_du_recensement',
       'organisme_recenseur', 'code_insee_du_canton',
       'code_insee_de_l_arrondissement', 'code_insee_du_departement',
       'code_insee_de_la_region', 'code_siren', 'codes_siren_des_epci',
       'code_postal', 'superficie_cadastrale', 'geometry', 'lon', 'lat'],
      dtype='object')

In [136]:

ref_comm = communes.copy()

ref_comm.drop(columns=['nom_officiel_en_majuscules','statut','date_du_recensement', 'organisme_recenseur'], inplace=True)
ref_comm.to_parquet("../" + REF_DIR + "ref_communes.parquet", index=False)
ref_comm.to_csv("../" + REF_DIR +"ref_communes.csv", index=False)

In [137]:
ref_comm.columns

Index(['cleabs', 'nom_officiel', 'code_insee', 'population',
       'code_insee_du_canton', 'code_insee_de_l_arrondissement',
       'code_insee_du_departement', 'code_insee_de_la_region', 'code_siren',
       'codes_siren_des_epci', 'code_postal', 'superficie_cadastrale',
       'geometry', 'lon', 'lat'],
      dtype='object')

In [139]:
ref_arrond = arrond.copy()
ref_arrond.head()

,cleabs,nom_officiel,nom_officiel_en_majuscules,numero_de_l_arrondissement_municipal,code_insee,code_insee_de_la_commune_de_rattach,code_postal,population,geometry,lon,lat
0,ARR_MUN_0000000000013207,Marseille 7e Arrondissement,MARSEILLE 7E ARRONDISSEMENT,07,13207,13055,13007,34866,"MULTIPOLYGON (((5.28741 43.26122, 5.28746 43.2...",5.359130,43.281466
1,ARR_MUN_0000000000075119,Paris 19e Arrondissement,PARIS 19E ARRONDISSEMENT,19,75119,75056,75019,178371,"MULTIPOLYGON (((2.40365 48.88168, 2.40371 48.8...",2.383082,48.887147
2,ARR_MUN_0000000000075118,Paris 18e Arrondissement,PARIS 18E ARRONDISSEMENT,18,75118,75056,75018,185825,"MULTIPOLYGON (((2.33517 48.90115, 2.33527 48.9...",2.348051,48.891809
3,ARR_MUN_0000000000075115,Paris 15e Arrondissement,PARIS 15E ARRONDISSEMENT,15,75115,75056,75015,228754,"MULTIPOLYGON (((2.29229 48.82712, 2.29219 48.8...",2.296380,48.841610
4,ARR_MUN_0000000000075114,Paris 14e Arrondissement,PARIS 14E ARRONDISSEMENT,14,75114,75056,75014,137581,"MULTIPOLYGON (((2.33486 48.84025, 2.33557 48.8...",2.324014,48.829624


In [140]:
# Reprojeter en WGS84
gdf_comm = ref_comm.to_crs(epsg=4326)
gdf_arrond = ref_arrond.to_crs(epsg=4326)

ref_dept = departements.copy()
gdf_dept = ref_dept.to_crs(epsg=4326)

# Harmoniser les colonnes
gdf_communes_clean = gdf_comm[[
    'code_insee', 'nom_officiel', 
    'population',
    'codes_siren_des_epci',
    'code_insee_du_departement',
    'code_insee_de_la_region',
    'geometry',
    'superficie_cadastrale',
    'lon',
    'lat'
]].rename(columns={'nom_officiel': 'nom_commune', 'superficie_cadastrale': 'superficie'})

gdf_arron_clean = gdf_arrond[[
    'code_insee', 'nom_officiel',
    'population',
    'geometry',
    'lon',
    'lat'
]].rename(columns={'nom_officiel': 'nom_commune'})

# Ajouter dept/région aux arrondissements via le code INSEE
# Paris 75101 → département 75
gdf_arron_clean['code_insee_du_departement'] = \
    gdf_arron_clean['code_insee'].str[:2]
    
# Créer un dictionnaire département → région
dept_to_region = gdf_dept.set_index('code_insee')['code_insee_de_la_region'].to_dict()

# {
#   '75': '11',  # Paris → Île-de-France
#   '59': '32',  # Nord → Hauts-de-France
#   ...
# }

# Appliquer aux arrondissements
gdf_arron_clean['code_insee_de_la_region'] = gdf_arron_clean['code_insee_du_departement'].map(dept_to_region)

# Combiner
gdf_final = pd.concat([gdf_communes_clean, gdf_arron_clean], ignore_index=True)

print(f"✅ Total : {len(gdf_final)} entités")

# Vérifier Paris
paris = gdf_final[gdf_final['code_insee'].str.startswith('751')]
print(f"Paris : {len(paris)} arrondissements")

✅ Total : 34791 entités
Paris : 20 arrondissements


In [141]:
gdf_final.head()

,code_insee,nom_commune,population,codes_siren_des_epci,code_insee_du_departement,code_insee_de_la_region,geometry,superficie,lon,lat
0,01001,L'Abergement-Clémenciat,859,200069193,01,84,"MULTIPOLYGON (((4.95841 46.15327, 4.95812 46.1...",1590.0,4.934111,46.151674
1,01002,L'Abergement-de-Varey,273,240100883,01,84,"MULTIPOLYGON (((5.4302 45.98277, 5.43012 45.98...",920.0,5.424073,46.007004
2,01004,Ambérieu-en-Bugey,15554,240100883,01,84,"MULTIPOLYGON (((5.40882 45.94206, 5.4085 45.94...",2460.0,5.367885,45.957404
3,01005,Ambérieux-en-Dombes,1917,200042497,01,84,"MULTIPOLYGON (((4.94298 45.97962, 4.94257 45.9...",1590.0,4.914417,45.999403
4,01006,Ambléon,114,200040350,01,84,"MULTIPOLYGON (((5.57083 45.75338, 5.57219 45.7...",590.0,5.593003,45.748372


In [142]:
gdf_paris = gdf_final[gdf_final['code_insee'].str.startswith('75')]

In [143]:
gdf_paris.head()

,code_insee,nom_commune,population,codes_siren_des_epci,code_insee_du_departement,code_insee_de_la_region,geometry,superficie,lon,lat
29194,75056,Paris,2113705,200054781,75,11,"MULTIPOLYGON (((2.33517 48.90115, 2.33527 48.9...",10540.0,2.320085,48.859054
34747,75119,Paris 19e Arrondissement,178371,NaN,75,11,"MULTIPOLYGON (((2.40365 48.88168, 2.40371 48.8...",NaN,2.383082,48.887147
34748,75118,Paris 18e Arrondissement,185825,NaN,75,11,"MULTIPOLYGON (((2.33517 48.90115, 2.33527 48.9...",NaN,2.348051,48.891809
34749,75115,Paris 15e Arrondissement,228754,NaN,75,11,"MULTIPOLYGON (((2.29229 48.82712, 2.29219 48.8...",NaN,2.296380,48.841610
34750,75114,Paris 14e Arrondissement,137581,NaN,75,11,"MULTIPOLYGON (((2.33486 48.84025, 2.33557 48.8...",NaN,2.324014,48.829624


In [144]:
gdf_ref_app_final = gdf_final.copy()

In [145]:
refapp_comm = gdf_final[["code_insee", "nom_commune"]].copy() 
refapp_comm["code"] = refapp_comm["code_insee"].astype(str) 
refapp_comm["code_type"] = "INSEE"
refapp_comm["type"] = "comm" 
refapp_comm["nom"] = refapp_comm["nom_commune"] 
refapp_comm = refapp_comm[["code", "code_type", "type", "nom"]]

refapp_comm.head()

,code,code_type,type,nom
0,01001,INSEE,comm,L'Abergement-Clémenciat
1,01002,INSEE,comm,L'Abergement-de-Varey
2,01004,INSEE,comm,Ambérieu-en-Bugey
3,01005,INSEE,comm,Ambérieux-en-Dombes
4,01006,INSEE,comm,Ambléon


In [147]:
epci.columns

Index(['cleabs', 'nom_officiel', 'nom_officiel_en_majuscules', 'nature',
       'codes_insee_des_communes_membres',
       'codes_insee_des_departements_membres', 'code_siren', 'geometry', 'lon',
       'lat'],
      dtype='object')

In [148]:
epci.head()

,cleabs,nom_officiel,nom_officiel_en_majuscules,nature,codes_insee_des_communes_membres,codes_insee_des_departements_membres,code_siren,geometry,lon,lat
0,EPCI____0000000200000172,CC Faucigny-Glières,CC FAUCIGNY-GLIERES,Communauté de communes,74024/74042/74049/74087/74164/74212/74312,74,200000172,"MULTIPOLYGON (((6.45511 46.05427, 6.45502 46.0...",6.428007,46.039487
1,EPCI____0000000200000438,CC du Pays de Pontchâteau Saint-Gildas-des-Bois,CC DU PAYS DE PONTCHATEAU SAINT-GILDAS-DES-BOIS,Communauté de communes,44050/44053/44068/44098/44129/44152/44161/4418...,44,200000438,"MULTIPOLYGON (((-1.98138 47.472, -1.98129 47.4...",-2.111359,47.475060
2,EPCI____0000000200000545,CC des Portes de Romilly-sur-Seine,CC DES PORTES DE ROMILLY-SUR-SEINE,Communauté de communes,10114/10164/10220/10280/10323/10341,10,200000545,"MULTIPOLYGON (((3.70283 48.47208, 3.70041 48.4...",3.714854,48.497358
3,EPCI____0000000200000628,CC Rhône Lez Provence,CC RHONE LEZ PROVENCE,Communauté de communes,84019/84063/84064/84078/84083,84,200000628,"MULTIPOLYGON (((4.77878 44.2107, 4.77819 44.21...",4.738389,44.246116
4,EPCI____0000000200000800,CC Cœur de Sologne,CC COEUR DE SOLOGNE,Communauté de communes,41036/41046/41106/41161/41251/41296,41,200000800,"MULTIPOLYGON (((2.08755 47.58796, 2.08697 47.5...",2.023303,47.573209


In [149]:
refapp_epci = epci[["code_siren", "nom_officiel"]].copy() 
refapp_epci["code"] = refapp_epci["code_siren"].astype(str) 
refapp_epci["code_type"] = "code_siren"
refapp_epci["type"] = "epci" 
refapp_epci["nom"] = refapp_epci["nom_officiel"] 
refapp_epci = refapp_epci[["code", "code_type", "type", "nom"]]

refapp_epci.head()


,code,code_type,type,nom
0,200000172,code_siren,epci,CC Faucigny-Glières
1,200000438,code_siren,epci,CC du Pays de Pontchâteau Saint-Gildas-des-Bois
2,200000545,code_siren,epci,CC des Portes de Romilly-sur-Seine
3,200000628,code_siren,epci,CC Rhône Lez Provence
4,200000800,code_siren,epci,CC Cœur de Sologne


In [150]:
departements.head()

,cleabs,nom_officiel,nom_officiel_en_majuscules,code_insee,code_insee_de_la_region,code_siren,geometry,lon,lat
0,DEPARTEM0000000000000063,Puy-de-Dôme,PUY-DE-DOME,63,84,226300010,"MULTIPOLYGON (((2.49665 45.55686, 2.49657 45.5...",3.083912,45.771855
1,DEPARTEM0000000000000059,Nord,NORD,59,32,225900018,"MULTIPOLYGON (((3.02003 50.15366, 3.02053 50.1...",3.088496,50.529112
2,DEPARTEM0000000000000061,Orne,ORNE,61,28,226100014,"MULTIPOLYGON (((-0.82109 48.47611, -0.82121 48...",0.044042,48.576460
3,DEPARTEM0000000000000013,Bouches-du-Rhône,BOUCHES-DU-RHONE,13,93,221300015,"MULTIPOLYGON (((5.2289 43.19806, 5.22891 43.19...",5.376624,43.542209
4,DEPARTEM0000000000000075,Paris,PARIS,75,11,227500055,"MULTIPOLYGON (((2.33517 48.90115, 2.33527 48.9...",2.320085,48.859054


In [151]:
refapp_dept = departements[["code_insee", "nom_officiel"]].copy() 
refapp_dept["code"] = refapp_dept["code_insee"].astype(str)
refapp_dept["code_type"] = "code_insee"
refapp_dept["type"] = "dept" 
refapp_dept["nom"] = refapp_dept["nom_officiel"] 
refapp_dept = refapp_dept[["code", "code_type", "type", "nom"]]

refapp_dept.head()

,code,code_type,type,nom
0,63,code_insee,dept,Puy-de-Dôme
1,59,code_insee,dept,Nord
2,61,code_insee,dept,Orne
3,13,code_insee,dept,Bouches-du-Rhône
4,75,code_insee,dept,Paris


In [152]:
regions = add_coord(regions)
regions.head()

,cleabs,nom_officiel,nom_officiel_en_majuscules,code_insee,code_siren,geometry,lon,lat
0,REGION__0000000000000024,Centre-Val de Loire,CENTRE-VAL DE LOIRE,24,234500023,"MULTIPOLYGON (((1.62329 48.74003, 1.62291 48.7...",1.716113,47.643887
1,REGION__0000000000000053,Bretagne,BRETAGNE,53,233500016,"MULTIPOLYGON (((-5.00711 48.41601, -5.0071 48....",-2.668907,48.155696
2,REGION__0000000000000044,Grand Est,GRAND EST,44,200052264,"MULTIPOLYGON (((3.64051 48.18462, 3.64011 48.1...",5.759323,48.794784
3,REGION__0000000000000075,Nouvelle-Aquitaine,NOUVELLE-AQUITAINE,75,200053759,"MULTIPOLYGON (((-1.74944 43.38256, -1.74943 43...",0.104648,44.976923
4,REGION__0000000000000076,Occitanie,OCCITANIE,76,200053791,"MULTIPOLYGON (((3.16399 42.45062, 3.16401 42.4...",2.117500,43.689861


In [153]:
refapp_reg = regions[["code_insee", "nom_officiel"]].copy() 
refapp_reg["code"] = refapp_reg["code_insee"].astype(str)
refapp_reg["code_type"] = "code_insee"
refapp_reg["type"] = "reg" 
refapp_reg["nom"] = refapp_reg["nom_officiel"] 
refapp_reg = refapp_reg[["code", "code_type", "type", "nom"]]

refapp_reg.head()

,code,code_type,type,nom
0,24,code_insee,reg,Centre-Val de Loire
1,53,code_insee,reg,Bretagne
2,44,code_insee,reg,Grand Est
3,75,code_insee,reg,Nouvelle-Aquitaine
4,76,code_insee,reg,Occitanie


In [155]:
collectivites.head()

,cleabs,nom_officiel,nom_officiel_en_majuscules,code_insee,code_insee_de_la_region,code_siren,geometry,lon,lat
0,COLLTERR000000000000031D,Haute-Garonne,HAUTE-GARONNE,31D,76,None,"MULTIPOLYGON (((1.6255 43.8018, 1.6265 43.8013...",0.972703,43.305290
1,COLLTERR000000000000032D,Gers,GERS,32D,76,None,"MULTIPOLYGON (((-0.2403 43.89795, -0.24008 43....",0.410079,43.695387
2,COLLTERR000000000000071D,Saône-et-Loire,SAONE-ET-LOIRE,71D,27,None,"MULTIPOLYGON (((4.92927 46.50416, 4.92918 46.5...",4.558680,46.656049
3,COLLTERR000000000000054D,Meurthe-et-Moselle,MEURTHE-ET-MOSELLE,54D,44,None,"MULTIPOLYGON (((5.73185 48.7246, 5.72952 48.72...",5.986972,48.956094
4,COLLTERR000000000000017D,Charente-Maritime,CHARENTE-MARITIME,17D,75,None,"MULTIPOLYGON (((-1.2331 45.80662, -1.2331 45.8...",-0.823487,45.730091


In [ ]:
# Collectivité Territoriales abndonnées temporairement
# refapp_coll = collectivites[["code_insee", "nom_officiel"]].copy() 
# refapp_coll["code"] = refapp_coll["code_insee"].astype(str)
# refapp_coll["code_type"] = "code_insee"
# refapp_coll["type"] = "reg" 
# refapp_coll["nom"] = refapp_coll["nom_officiel"] 
# refapp_coll["communes"] = ""
# refapp_refapp_collreg = refapp_coll[["code", "code_type", "type", "nom"]]
# 
# refapp_coll.head()

In [156]:
print("Chargement Zone d'emploi ...")
import pandas as pd
import os

file_name = "ZE2020_au_01-01-2025.xlsx"

file_path = os.path.join("..", RAW_DIR ,'territoires', file_name)
#file_path = '..' / RAW_DIR / 'territoires' / file_name
print(f"File path: {file_path}")

df_ze = pd.read_excel(
    file_path,
    sheet_name='ZE2020',
    skiprows=5,
    dtype={'ZE2020': str}
)

Chargement Zone d'emploi ...
File path: ../data/raw/territoires/ZE2020_au_01-01-2025.xlsx


In [117]:
#df_ze = add_coord(df_ze)
df_ze.head()

,ZE2020,LIBZE2020,NB_COM
0,0051,Alençon,184
1,0052,Arles,20
2,0053,Avignon,36
3,0054,Beauvais,327
4,0055,Bollène-Pierrelatte,29


In [118]:
refapp_ze = df_ze[["ZE2020", "LIBZE2020"]].copy() 
refapp_ze["code"] = refapp_ze["ZE2020"].astype(str)
refapp_ze["code_type"] = "INSEE"
refapp_ze["type"] = "ze" 
refapp_ze["nom"] = refapp_ze["LIBZE2020"] 
refapp_ze = refapp_ze[["code", "code_type", "type", "nom"]]

refapp_ze.head()

,code,code_type,type,nom
0,0051,INSEE,ze,Alençon
1,0052,INSEE,ze,Arles
2,0053,INSEE,ze,Avignon
3,0054,INSEE,ze,Beauvais
4,0055,INSEE,ze,Bollène-Pierrelatte


In [157]:
# Rassembler les df pour le rérentiel des territoires (collectivité territopriale exclue pour le moment)
# concat avec axis=0 par défaut => empile les lignes
refapp_final = pd.concat([refapp_comm, refapp_dept, refapp_epci, refapp_reg, refapp_ze])

refapp_final.shape

(36449, 4)

In [171]:
refapp_final.head()

,code,code_type,type,nom
0,01001,INSEE,comm,L'Abergement-Clémenciat
1,01002,INSEE,comm,L'Abergement-de-Varey
2,01004,INSEE,comm,Ambérieu-en-Bugey
3,01005,INSEE,comm,Ambérieux-en-Dombes
4,01006,INSEE,comm,Ambléon


In [158]:
import os
filepath = os.path.join("..", REF_DIR , "ref_app_territoires.parquet")
#refapp_final.to_parquet(".." / REF_DIR / "ref_app_territoires.parquet", index=False)

refapp_final.to_parquet(filepath, index=False)

### Construire les fichiers avec geometries simplifiées 

In [172]:
gdf_comm = communes[[
    "code_insee",
    "nom_officiel",
    "geometry",
    "lon",
    "lat"
]].copy()

In [173]:
gdf_dept = departements[[
    "code_insee",
    "nom_officiel",
    "geometry",
    "lon",
    "lat"
]].copy()

In [174]:
gdf_epci = epci[[
    "code_siren",
    "nom_officiel",
    "geometry",
    "lon",
    "lat"
]].copy()


In [177]:
gdf_reg = regions[[
    "code_siren",
    "nom_officiel",
    "geometry",
    "lon",
    "lat"
]].copy()


In [178]:
gdf_coll = collectivites[[
    "code_siren",
    "nom_officiel",
    "geometry",
    "lon",
    "lat"
]].copy()

In [179]:
print(gdf_comm.crs)
print(gdf_dept.crs) 
print(gdf_epci.crs)
print(gdf_reg.crs)
print(gdf_coll.crs)

EPSG:4326
EPSG:4326
EPSG:4326
EPSG:4326
EPSG:4326


In [180]:
gdf_ref_app_final.head()

,code_insee,nom_commune,population,codes_siren_des_epci,code_insee_du_departement,code_insee_de_la_region,geometry,superficie,lon,lat
0,01001,L'Abergement-Clémenciat,859,200069193,01,84,"MULTIPOLYGON (((4.95841 46.15327, 4.95812 46.1...",1590.0,4.934111,46.151674
1,01002,L'Abergement-de-Varey,273,240100883,01,84,"MULTIPOLYGON (((5.4302 45.98277, 5.43012 45.98...",920.0,5.424073,46.007004
2,01004,Ambérieu-en-Bugey,15554,240100883,01,84,"MULTIPOLYGON (((5.40882 45.94206, 5.4085 45.94...",2460.0,5.367885,45.957404
3,01005,Ambérieux-en-Dombes,1917,200042497,01,84,"MULTIPOLYGON (((4.94298 45.97962, 4.94257 45.9...",1590.0,4.914417,45.999403
4,01006,Ambléon,114,200040350,01,84,"MULTIPOLYGON (((5.57083 45.75338, 5.57219 45.7...",590.0,5.593003,45.748372


In [181]:

gdf_app_comm =  gdf_ref_app_final[[
    "code_insee",
    "nom_commune",
    "geometry",
    "lon",
    "lat"
]].copy()

In [182]:
gdf_app_comm[gdf_app_comm['code_insee'].str.startswith('75')].head()

,code_insee,nom_commune,geometry,lon,lat
29194,75056,Paris,"MULTIPOLYGON (((2.33517 48.90115, 2.33527 48.9...",2.320085,48.859054
34747,75119,Paris 19e Arrondissement,"MULTIPOLYGON (((2.40365 48.88168, 2.40371 48.8...",2.383082,48.887147
34748,75118,Paris 18e Arrondissement,"MULTIPOLYGON (((2.33517 48.90115, 2.33527 48.9...",2.348051,48.891809
34749,75115,Paris 15e Arrondissement,"MULTIPOLYGON (((2.29229 48.82712, 2.29219 48.8...",2.296380,48.841610
34750,75114,Paris 14e Arrondissement,"MULTIPOLYGON (((2.33486 48.84025, 2.33557 48.8...",2.324014,48.829624


In [183]:
comm_s = simplify_geom(gdf_comm, tolerance=70)
dept_s = simplify_geom(gdf_dept, tolerance=80)
epci_s = simplify_geom(gdf_epci, tolerance=80)
reg_s = simplify_geom(gdf_reg, tolerance=100)
coll_s = simplify_geom(gdf_coll, tolerance=100)
# app_comm_s : geometry simpplifiée par l'app contenant comm + dept
app_comm_s = simplify_geom(gdf_app_comm, tolerance=70)

In [185]:
print('comms_s:',comm_s.columns)
print('dept_s:',dept_s.columns)
print('epci_s:',epci_s.columns)
print('reg_s:',reg_s.columns)
print('coll_s:',coll_s.columns)
print('app_comm_s:',app_comm_s.columns)

comms_s: Index(['code_insee', 'nom_officiel', 'geometry', 'lon', 'lat'], dtype='object')
dept_s: Index(['code_insee', 'nom_officiel', 'geometry', 'lon', 'lat'], dtype='object')
epci_s: Index(['code_siren', 'nom_officiel', 'geometry', 'lon', 'lat'], dtype='object')
reg_s: Index(['code_siren', 'nom_officiel', 'geometry', 'lon', 'lat'], dtype='object')
coll_s: Index(['code_siren', 'nom_officiel', 'geometry', 'lon', 'lat'], dtype='object')
app_comm_s: Index(['code_insee', 'nom_commune', 'geometry', 'lon', 'lat'], dtype='object')


In [ ]:

app_comm_s.to_parquet('../data/processed/geom/app_communes_geom_simplified.parquet')
gdf_app_comm.to_parquet('../data/processed/geom/app_communes_geom_original.parquet')
# La simplication geométrique a fait passer le fichier parquet de 417Mo à 24Mo
# -rw-r--r--  1 jean-jacques  staff   417M Feb 28 22:38 app_communes_geom_original.parquet
# -rw-r--r--  1 jean-jacques  staff    24M Feb 28 22:38 app_communes_geom_simplified.parquet

In [ ]:
geom_path = os.path.join("..", GEOM_DIR)

comm_s.to_parquet(geom_path + "communes_geom_simplified.parquet", index=False)
dept_s.to_parquet(geom_path + "departements_geom_simplified.parquet", index=False)
epci_s.to_parquet(geom_path + "epci_geom_simplified.parquet", index=False)
reg_s.to_parquet(geom_path + "regions_geom_simplified.parquet", index=False)
coll_s.to_parquet(geom_path + "collectivites_geom_simplified.parquet", index=False)


In [195]:
ls -lh ../data/processed/geom/

total 1063760
-rw-r--r--  1 jean-jacques  staff   417M Feb 28 22:38 app_communes_geom_original.parquet
-rw-r--r--  1 jean-jacques  staff    24M Feb 28 22:38 app_communes_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff   1.4M Feb 28 22:04 collectivites_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff   934K Feb 25 19:10 collectivites_topo.json
-rw-r--r--  1 jean-jacques  staff    24M Feb 28 22:04 communes_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff    28M Feb 25 18:51 communes_topo.json
-rw-r--r--  1 jean-jacques  staff   1.6M Feb 28 22:04 departements_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff   935K Feb 25 18:59 departements_topo.json
-rw-r--r--  1 jean-jacques  staff   5.2M Feb 28 22:04 epci_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff   3.1M Feb 25 19:09 epci_topo.json
-rw-r--r--  1 jean-jacques  staff   619K Feb 28 22:04 regions_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff   457K Feb 25 18:59 regions_topo.jso

In [193]:
# vérification que les arrondissements sont bien dans le référentiel des communes pour l'app
import pandas as pd
df = pd.read_parquet('/Users/jean-jacques/code/jjchabutDataCRM/sante-territoires/data/processed/ref/ref_app_communes.parquet')
print(df.shape)
print(df.columns.tolist())
print(df.head(20).to_string())


(21, 10)
['code_insee', 'nom_commune', 'population', 'codes_siren_des_epci', 'code_insee_du_departement', 'code_insee_de_la_region', 'geometry', 'superficie', 'lon', 'lat']
      code_insee               nom_commune  population codes_siren_des_epci code_insee_du_departement code_insee_de_la_region                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              